In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType
from datetime import datetime

In [0]:
# ✅ 표시 기준 (저장은 여전히 UTC) 
spark.conf.set("spark.sql.session.timeZone", "Asia/Seoul")

In [0]:
# =========================================================
# 0. Job watermark window
# =========================================================
next_start = datetime.fromisoformat(
    dbutils.jobs.taskValues.get("check_lasttime", "next_start")
)
next_end = datetime.fromisoformat(
    dbutils.jobs.taskValues.get("check_lasttime", "next_end")
)

In [0]:
# =========================================================
# 1. BRZ DAILY temp 로드
# =========================================================
raw_df = spark.table(
    "hive_metastore.demo_airstatus_bronze.BRZ_temp_seoul_air_quality_6h"
)

In [0]:
# =========================================================
# 2. dataTime 문자열 정제 (MONTH와 동일한 방식 ✅)
#    - "-" 제거
#    - "24:00" → 다음날 "00:00"
#    - KST 기준 → UTC timestamp
# =========================================================
clean_df = (
    raw_df
    # ❌ 시간 없는 placeholder 제거
    .filter(col("dataTime") != "-")

    .withColumn(
        "dataTime_fixed",
        when(
            col("dataTime").endswith(" 24:00"),
            to_utc_timestamp(
                concat(
                    date_add(to_date(substring(col("dataTime"), 1, 10)), 1),
                    lit(" 00:00")
                ),
                "Asia/Seoul"
            )
        ).otherwise(
            to_utc_timestamp(col("dataTime"), "Asia/Seoul")
        )
    )
)

In [0]:
# =========================================================
# 2-1. dataTime_fixed에 9시간 추가 (UTC→KST 보정)
# =========================================================
clean_df = clean_df.withColumn(
    "dataTime_fixed",
    col("dataTime_fixed") + expr("INTERVAL 9 HOURS")
)

In [0]:
# =========================================================
# 3. 값 컬럼 정제
# =========================================================
clean_df = (
    clean_df
    .withColumn(
        "khaiValue",
        coalesce(regexp_replace(col("khaiValue"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "khaiGrade",
        coalesce(regexp_replace(col("khaiGrade"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "pm10Value",
        coalesce(regexp_replace(col("pm10Value"), "-", "").cast(IntegerType()), lit(0))
    )
    .withColumn(
        "pm25Value",
        coalesce(regexp_replace(col("pm25Value"), "-", "").cast(IntegerType()), lit(0))
    )
)

In [0]:
# =========================================================
# 4. dataTime 컬럼 확정 (AMBIGUOUS 방지 ✅)
# =========================================================
final_df = (
    clean_df
    .select(
        "stationName",
        col("dataTime_fixed").alias("dataTime"),
        "khaiValue",
        "khaiGrade",
        "pm10Value",
        "pm25Value"
    )
    .filter(col("dataTime").isNotNull())
)

In [0]:
# =========================================================
# 5. NULL 검증 (기존 정책 유지)
# =========================================================
null_check_df = (
    final_df
    .groupBy("stationName")
    .agg(
        count("*").alias("total_cnt"),
        sum(when(col("dataTime").isNull(), 1).otherwise(0)).alias("null_cnt")
    )
)

bad_station_df = null_check_df.filter(col("null_cnt") >= 2)

if bad_station_df.count() > 0:
    (
        bad_station_df
        .withColumn("window_start", lit(next_start))
        .withColumn("window_end", lit(next_end))
        .withColumn("issue_type", lit("MULTIPLE_NULL_DATETIME"))
        .withColumn("logged_at", current_timestamp())
        .write
        .mode("append")
        .format("delta")
        .saveAsTable(
            "hive_metastore.demo_airstatus_silver.SLV_data_quality_log"
        )
    )
    raise Exception("❌ NULL dataTime >= 2 detected. Stop pipeline.")


In [0]:
# =========================================================
# 6. 중복 제거
# =========================================================
final_df = final_df.dropDuplicates(["stationName", "dataTime"])

In [0]:
# =========================================================
# 7. 파생 컬럼
# =========================================================
fact_df = (
    final_df
    .withColumn("year", year("dataTime"))
    .withColumn("month", month("dataTime"))
    .withColumn("day", dayofmonth("dataTime"))
    .withColumn("hour", hour("dataTime"))
)

In [0]:
# =========================================================
# 8. Silver DAILY temp / main 저장
# =========================================================
fact_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_silver.SLV_temp_fact_air_quality_6h"
    )

fact_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_silver.SLV_fact_air_quality"
    )

print("✅ DJ_SLV_데이터전처리 (DAILY) completed successfully.")